In [1]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict

In [2]:
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

In [3]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [7]:
def create_outline(state: BlogState) -> BlogState:
    
    # Fetch title
    title = state['title']

    # Call llm gen outline
    prompt = f"Generate a detailed outline for a blog on the topic - {title}"
    chain = llm | StrOutputParser()

    # Update state
    state['outline'] = chain.invoke(prompt)

    return state

In [8]:
def create_blog(state: BlogState) -> BlogState:

    # Fetch title and outline
    title = state['title']
    outline = state['outline']

    # form a prompt
    prompt = f"Write a detailed blog on the title - {title} using the following outline \n {outline}"

    chain = llm | StrOutputParser()
    state['content'] = chain.invoke(prompt)

    return state

In [10]:
graph = StateGraph(BlogState)

# Add Nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# Add Edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

# Compile
workflow = graph.compile()

In [12]:
initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)
print(final_state)
print(final_state['outline'])

{'title': 'Rise of AI in India', 'outline': 'Here is a detailed, SEO-friendly outline for a blog post on the **"Rise of AI in India."** \n\n---\n\n# Blog Title Options:\n1. *The AI Revolution in India: How Artificial Intelligence is Shaping the Future*\n2. *From Code to Catalyst: The Meteoric Rise of AI in India*\n3. *Bharat 2.0: Understanding India’s Artificial Intelligence Boom*\n\n---\n\n## **I. Introduction**\n* **The Hook:** Briefly paint a picture of India’s rapid technological transformation—shifting from an IT back-office to a global hub for deep-tech innovation.\n* **The Context:** Mention the convergence of affordable internet, a massive young population, government backing, and a thriving startup ecosystem.\n* **Thesis Statement:** AI in India is no longer just a futuristic concept; it is an active economic engine driving growth, solving grassroots problems, and positioning India as a global AI superpower.\n\n---\n\n## **II. Key Drivers Behind India’s AI Boom**\n* **A. Demog